# Building a small autoresearch loop

We will train a small language model, try one change ourselves, and then let a coding agent investigate. Every experiment starts from scratch and uses the same data, training budget, and evaluation.

The research question is whether we can improve predictions on held-out text within **ten seconds of CPU training**. We will follow Question → Hypothesis → Experiment → Analyze results, with each result informing what to try next.

This is a small teaching implementation inspired by [Karpathy’s autoresearch](https://github.com/karpathy/autoresearch). His published GPU run uses a different model and a five-minute training budget.

## Open the actual experiment

This notebook runs the files in this folder. `train.py` contains the model, optimizer, and batch size. `experiment.py` and `prepare.py` fix the experimental conditions and evaluation. The coding agent will use these same files.

Run the cells in order. Leave the original training file unchanged until the manual experiment.

In [ ]:
from pathlib import Path
from datetime import datetime
import csv
import hashlib
import importlib
import json
import subprocess

from IPython.display import Code, Markdown, display

ROOT = Path.cwd()
for name in ['train.py', 'experiment.py', 'prepare.py', 'program.md', 'data/input.txt']:
    assert (ROOT / name).is_file(), f'Open this notebook inside the workshop folder: missing {name}'

TRAIN_PATH = ROOT / 'train.py'
starting_source = TRAIN_PATH.read_bytes()
assert starting_source.count(b'BATCH_SIZE = 32') == 1, 'Start this walkthrough with the original batch size of 32.'
expected_manual_source = starting_source.replace(b'BATCH_SIZE = 32', b'BATCH_SIZE = 16', 1)
starting_sha = hashlib.sha256(starting_source).hexdigest()
print('Workshop folder:', ROOT)
print('Starting train.py SHA:', starting_sha)


## The dataset

The bundled corpus is tiny Shakespeare. We use the first 90% of its bytes for training and the final 10% for validation. Each example contains **32 input bytes** and one target: the byte immediately after that context.

The target is outside the input. We ask the model to assign probabilities to the 256 possible byte values, then measure the probability it assigned to the actual next byte.

In [ ]:
from prepare import read_verified_data

corpus = read_verified_data()
split = int(0.9 * len(corpus))
print(f'Corpus: {len(corpus):,} bytes')
print(f'Training: {split:,} bytes; validation: {len(corpus) - split:,} bytes')
print('Example context:', repr(corpus[:32].decode('utf-8', errors='replace')))
print('Actual next byte:', repr(corpus[32:33].decode('utf-8', errors='replace')))


## The model

Our starting model is a transformer with **85,056 parameters**, one layer, four attention heads, and a hidden width of 64. It receives 32 bytes and produces 256 logits for the next byte. AdamW updates its parameters during training.

This is the complete editable training file. The agent can change the model, optimizer, or batch size while preserving the evaluator’s input and output interface.

In [ ]:
import torch
import train

importlib.reload(train)
model = train.build_model(256, 32)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
assert parameter_count == 85056, f'This walkthrough expects the original model, found {parameter_count:,} parameters.'
print(f'{parameter_count:,} parameters')
print('Input: [batch, 32] byte indices → output: [batch, 256] logits')
display(Code(starting_source.decode('utf-8'), language='python'))


## What stays fixed

Each subprocess initializes a fresh model with seed 42 and trains on two CPU threads for ten seconds. Startup and evaluation take additional time.

The evaluator checks 4,096 fixed windows from the validation split. It averages the negative log probability of the correct next byte, measured in bits: **validation bits per byte, or BPB**. Lower is better. A uniform prediction over all 256 byte values would cost eight bits per byte.

We leave the data, split, context length, seed, training budget, and evaluator unchanged. Otherwise the comparison could answer a different question.

In [ ]:
import experiment as fixed_experiment

print({
    'seed': fixed_experiment.SEED,
    'CPU threads': fixed_experiment.CPU_THREADS,
    'context bytes': fixed_experiment.CONTEXT_LENGTH,
    'training seconds': fixed_experiment.TRAIN_SECONDS,
    'validation windows': fixed_experiment.VALIDATION_EXAMPLES,
})


## Run the baseline

This executes `experiment.py` in a fresh subprocess. It writes a new result file with the score, training steps, parameter count, and training-code SHA. We use unique filenames so rerunning a cell cannot overwrite earlier evidence.

In [ ]:
def run_experiment(label):
    stamp = datetime.now().strftime('%Y%m%d-%H%M%S-%f')
    output = ROOT / 'runs' / f'live-{label}-{stamp}.json'
    command = ['uv', 'run', 'experiment.py', '--output', str(output)]
    print('Running:', ' '.join(command))
    completed = subprocess.run(command, cwd=ROOT, text=True, capture_output=True)
    if completed.returncode:
        print(completed.stdout)
        print(completed.stderr)
        completed.check_returncode()
    result = json.loads(output.read_text())
    display(result)
    print('Artifact:', output.relative_to(ROOT))
    return result

baseline = run_experiment('baseline')
assert baseline['candidate_sha256'] == starting_sha
assert baseline['batch_size'] == 32


## Make one change yourself

Our hypothesis is that smaller batches may help within this time budget by allowing more frequent parameter updates. They can also make each update noisier, so an improvement is not guaranteed.

**Your action:** open `train.py`, change `BATCH_SIZE = 32` to `BATCH_SIZE = 16`, and save. Make that edit yourself before running the next cell. Leave everything else unchanged.

The next cell verifies that exact edit before starting the experiment.

In [ ]:
current_source = TRAIN_PATH.read_bytes()
if current_source == starting_source:
    raise RuntimeError('No edit yet. Open train.py, change BATCH_SIZE = 32 to BATCH_SIZE = 16, save, and run this cell again.')
if current_source != expected_manual_source:
    raise RuntimeError('The file contains a different edit. Inspect it; this demonstration expects only BATCH_SIZE 32 → 16.')

manual = run_experiment('batch-16')
assert manual['batch_size'] == 16
assert manual['candidate_sha256'] == hashlib.sha256(expected_manual_source).hexdigest()


## Compare the evidence

Look at the score and how many training steps fit into the budget. The model and parameter count stayed the same; the batch size changed. One short run can suggest a direction, but a small difference needs repetition before we trust it.

In [ ]:
print(f"Baseline: batch {baseline['batch_size']}, BPB {baseline['val_bpb']:.6f}, {baseline['steps']} steps")
print(f"Manual:   batch {manual['batch_size']}, BPB {manual['val_bpb']:.6f}, {manual['steps']} steps")
print(f"BPB difference, manual minus baseline: {manual['val_bpb'] - baseline['val_bpb']:+.6f}")
print('Negative means the manual change scored better in this comparison.')
print('Parameter counts:', baseline['num_params'], '→', manual['num_params'])


## Restore our manual change

Before handing control to the agent, return `train.py` to the starting version. The following cell reverses only the exact batch-size edit we made. It leaves the measured artifacts and notebook outputs intact and refuses to overwrite any further changes.

In [ ]:
current_source = TRAIN_PATH.read_bytes()
if current_source == expected_manual_source:
    TRAIN_PATH.write_bytes(starting_source)
    print('Restored the original batch size of 32. Experiment artifacts are retained.')
elif current_source == starting_source:
    print('The original training file is already restored.')
else:
    raise RuntimeError('train.py contains additional edits. Left them intact; inspect before continuing.')

assert hashlib.sha256(TRAIN_PATH.read_bytes()).hexdigest() == starting_sha


## Let the coding agent investigate

Open your existing coding agent in this same local workshop folder. It already knows how to read files, edit code, and run commands. `program.md` connects those actions into a bounded research process.

Give it this prompt:

> Read `program.md` and carry out its bounded research session. Complete at most three candidate experiments. Edit only `train.py`, preserve the fixed evaluator, execute each experiment, and use the actual evidence to decide what to try next. Stop with an explanation of the results and one proposed next investigation.

Watch its hypothesis, code change, experiment, and decision. It should preserve candidate commits and JSON results, including unsuccessful attempts. It records the baseline and decisions in `results.tsv`. Notebook autosaves are allowed; protected experiment inputs stay unchanged.

**Pause here while the agent works.** An agent that describes an experiment without running it has not completed the investigation.

## Inspect the actual research record

Run this after the agent has produced results. The table reads `results.tsv` directly; it does not create example findings. Open the referenced JSON and code commit when assessing a conclusion.

In [ ]:
from html import escape
from IPython.display import HTML

record = ROOT / 'results.tsv'
if not record.exists():
    print('No results.tsv yet. Let the coding agent complete its experiment, then rerun this cell.')
else:
    with record.open(newline='') as handle:
        reader = csv.DictReader(handle, delimiter='	')
        fieldnames = reader.fieldnames or []
        rows = list(reader)
    if not rows:
        print('The research record has no experiment rows yet.')
    else:
        header = ''.join(f'<th>{escape(name)}</th>' for name in fieldnames)
        body = ''.join('<tr>' + ''.join(f'<td>{escape(str(row.get(name, ""))) or "—"}</td>' for name in fieldnames) + '</tr>' for row in rows)
        display(HTML('<div style="overflow-x:auto"><table><thead><tr>' + header + '</tr></thead><tbody>' + body + '</tbody></table></div>'))
        print('Rows shown:', len(rows), '— read from', record.name)


## Bring human knowledge into the next investigation

Now that you have followed the trajectory, ask the agent to explain a result and show its evidence. Which changes appear useful? What else changed at the same time? What would a repeat establish?

You may remember a relevant paper, notice an assumption the agent missed, or care about a constraint it has not considered. Explain why that knowledge matters and ask how it changes the next investigation. Keep the evaluation fixed when making a comparable follow-up.

The interface question for the challenge is how to make that continuing exchange easier: knowing what is happening, finding relevant prior knowledge at the right time, and helping someone understand enough evidence to ask a better question.

## Your investigation

Choose any research problem and tools. Demonstrate a completed investigation, evaluate the evidence, have the agent choose an informed next step, and make human participation or understanding meaningful. Negative results count. Papiers is optional.

These ten-second runs provide local evidence, not a general benchmark record. Repeatedly selecting changes against the same validation set can overfit it. Preserve the results and be explicit about what they do and do not establish.